# SVD Quantization on GPU

Run SVD-based sub-1-bit quantization with GPU acceleration.

**Runtime**: Runtime > Change runtime type > **GPU** (T4 or better)

**Expected time**: ~30-60 minutes depending on model size

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [ ]:
# Clone/pull the repo
!git clone https://github.com/toxzak-svg/Quantization-Exploration.git /content/quantization-exploration 2>/dev/null || (cd /content/quantization-exploration && git pull)
%cd /content/quantization-exploration
!git pull origin main

Already up to date.
/content/quantization-exploration
From https://github.com/toxzak-svg/Quantization-Exploration
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
# Install dependencies
!pip install torch transformers accelerate numpy scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [ ]:
# Download Gemma 2 2B model
from huggingface_hub import snapshot_download
from google.colab import userdata
from safetensors.torch import load_file, save_file
import os
import torch

token = userdata.get('HF_TOKEN')
MODEL_DIR = "/content/models/gemma-4-E2B"

snapshot_download(
    repo_id="google/gemma-2-2b-it",
    local_dir=MODEL_DIR,
    token=token
)

# Merge shards if script expects a single model.safetensors
target_path = os.path.join(MODEL_DIR, "model.safetensors")
if not os.path.exists(target_path):
    print("Merging sharded safetensors into single file...")
    shards = sorted([f for f in os.listdir(MODEL_DIR) if f.startswith("model-") and f.endswith(".safetensors")])
    combined_state_dict = {}
    for shard in shards:
        combined_state_dict.update(load_file(os.path.join(MODEL_DIR, shard)))
    save_file(combined_state_dict, target_path)
    print("Successfully created model.safetensors")

print(f"Files in {MODEL_DIR}:")
print(os.listdir(MODEL_DIR))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Merging sharded safetensors into single file...
Successfully created model.safetensors
Files in /content/models/gemma-4-E2B:
['generation_config.json', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model', 'README.md', 'model.safetensors', 'model-00001-of-00002.safetensors', '.gitattributes', '.cache', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json']


In [ ]:
# Run SVD quantization at 60% threshold with the patched script
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_60.pt \
    --threshold 0.60

Found 182 weights
Energy threshold: 0.6 (60%)
Layer 0: rank= 802 (35% of full), bpw=0.4352, shape=[2304,9216]
Layer 1: rank= 814 (35% of full), bpw=0.4417, shape=[9216,2304]
Layer 2: rank= 856 (37% of full), bpw=0.4645, shape=[9216,2304]
Layer 3: rank= 281 (27% of full), bpw=0.3966, shape=[1024,2304]
Layer 4: rank= 370 (18% of full), bpw=0.3414, shape=[2304,2048]
Layer 50: rank= 778 (34% of full), bpw=0.4222, shape=[9216,2304]
Layer 100: rank= 845 (37% of full), bpw=0.4585, shape=[9216,2304]
Layer 150: rank= 283 (28% of full), bpw=0.3995, shape=[1024,2304]

Results:
  Avg rank: 551
  Avg bits/weight: 0.4356
  Compression: 36.7x

Saved to quantized/gemma_svd_60.pt


In [ ]:
import os
import subprocess

# 1. Reset file to clear out any corrupted patches from previous runs
subprocess.run(["git", "restore", "scripts/eval_reconstruction.py"], cwd="/content/quantization-exploration")

eval_path = 'scripts/eval_reconstruction.py'
with open(eval_path, 'r') as f:
    content = f.read()

svd_helper = """
def reconstruct_from_svd(q_entry, device='cpu'):
    U = q_entry['U'].to(device)
    S = q_entry['S'].to(device)
    Vt = q_entry['Vt'].to(device)
    return torch.matmul(U * S, Vt)
"""

# 2. Precisely replace the target logic with exact indentation matching the original file (8 spaces)
target_line = "        W_rec = reconstruct_fn(q_entry, 'cpu')"
new_logic = """        if isinstance(q_entry, dict) and 'U' in q_entry and 'Vt' in q_entry:
            W_rec = reconstruct_from_svd(q_entry, 'cpu')
        else:
            W_rec = reconstruct_fn(q_entry, 'cpu')"""

if target_line in content:
    content = content.replace(target_line, new_logic)
    content = content.replace("def main():", svd_helper + "\ndef main():")

    with open(eval_path, 'w') as f:
        f.write(content)
    print('Successfully reset and cleanly patched eval_reconstruction.py for SVD support.')
else:
    print('Could not find the target line to patch. The file might be structured differently than expected.')

Successfully reset and cleanly patched eval_reconstruction.py for SVD support.


In [ ]:
# Also try 70% threshold for comparison
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_70.pt \
    --threshold 0.70

Found 182 weights
Energy threshold: 0.7 (70%)
Layer 0: rank=1035 (45% of full), bpw=0.5616, shape=[2304,9216]
Layer 1: rank=1047 (45% of full), bpw=0.5681, shape=[9216,2304]
Layer 2: rank=1086 (47% of full), bpw=0.5893, shape=[9216,2304]
Layer 3: rank= 370 (36% of full), bpw=0.5222, shape=[1024,2304]
Layer 4: rank= 507 (25% of full), bpw=0.4678, shape=[2304,2048]
Layer 50: rank=1008 (44% of full), bpw=0.5470, shape=[9216,2304]
Layer 100: rank=1069 (46% of full), bpw=0.5801, shape=[9216,2304]
Layer 150: rank= 371 (36% of full), bpw=0.5237, shape=[1024,2304]

Results:
  Avg rank: 712
  Avg bits/weight: 0.5595
  Compression: 28.6x

Saved to quantized/gemma_svd_70.pt


In [ ]:
# Rerun evaluation with robust SVD detection and patched paths
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --max-layers 10

RECONSTRUCTION QUALITY EVALUATION

[1] Loading original weights...
  Loaded 0 weight matrices
  Evaluating first 0 layers

[2] Evaluating reconstruction quality...

  Skipping quantized/gemma_hybrid_stream.pt - not found

  Skipping quantized/gemma_ternary_aggressive.pt - not found

  Skipping quantized/gemma_magq.pt - not found

  Skipping quantized/gemma-4-E2B-sub1bit.pt - not found

SUMMARY - Ordered by Quality (best first)
Method                         Avg MSE      Max Error    Est. PPL    
----------------------------------------------------------------------
----------------------------------------------------------------------
Note: Estimated PPL assumes base Gemma 4 E2B perplexity ~5 on WikiText-2
      Actual perplexity testing requires full model loading (memory limited)


In [ ]:
# Copy results to Google Drive
import shutil
import os

os.makedirs("/content/drive/MyDrive/quantization-results", exist_ok=True)
for f in ["quantized/gemma_svd_60.pt", "quantized/gemma_svd_70.pt"]:
    if os.path.exists(f):
        shutil.copy(f, "/content/drive/MyDrive/quantization-results/")
        print(f"Copied {f}")

Copied quantized/gemma_svd_60.pt
Copied quantized/gemma_svd_70.pt


In [ ]:
import os

eval_path = '/content/quantization-exploration/scripts/eval_reconstruction.py'
with open(eval_path, 'r') as f:
    content = f.read()

# Create the robust SVD unpacking and reconstruction logic
svd_helper_new = """
def unpack_ternary(packed, shape, device='cpu'):
    packed = packed.to(torch.int32).to(device)
    powers = torch.tensor([81, 27, 9, 3, 1], dtype=torch.int32, device=device)
    encoded = (packed.unsqueeze(1) // powers) % 3
    encoded = encoded.flatten()

    import math
    n = math.prod(shape)
    encoded = encoded[:n]

    t = encoded.to(torch.float32) - 1.0
    return t.reshape(shape)

def reconstruct_from_svd(q_entry, device='cpu'):
    if 'U_packed' in q_entry:
        U = unpack_ternary(q_entry['U_packed'], q_entry['U_shape'], device) * q_entry['U_scale']
        Vt = unpack_ternary(q_entry['Vt_packed'], q_entry['Vt_shape'], device) * q_entry['Vt_scale']
        S_q = q_entry['S'].to(torch.float32).to(device)
        S_scale = q_entry['S_scale']
        sigma_bits = q_entry.get('sigma_bits', 2)
        qmax = 2 ** (sigma_bits - 1) - 1
        S = S_q * (S_scale / qmax)
        return torch.matmul(U * S, Vt)
    else:
        # Fallback to plain U, S, Vt
        U = q_entry.get('U', q_entry.get('u')).to(device)
        S = q_entry.get('S', q_entry.get('s')).to(device)
        Vt = q_entry.get('Vt', q_entry.get('vt', q_entry.get('v'))).to(device)
        return torch.matmul(U * S, Vt)
"""

# Let's replace the old svd helper if it exists, or inject the new one
import re
content = re.sub(r'def reconstruct_from_svd.*?(?=\ndef main\(\):|\ndef )', svd_helper_new, content, flags=re.DOTALL)

# Make sure it's injected
if 'unpack_ternary' not in content:
    content = content.replace("def main():", svd_helper_new + "\ndef main():")

# Ensure the route logic is still there
new_route = """        if isinstance(q_entry, dict) and ('U_packed' in q_entry or 'U' in q_entry or 'u' in q_entry):
            W_rec = reconstruct_from_svd(q_entry, 'cpu')
        else:
            W_rec = reconstruct_fn(q_entry, 'cpu')"""
old_route = """        if isinstance(q_entry, dict) and 'U' in q_entry and 'Vt' in q_entry:
            W_rec = reconstruct_from_svd(q_entry, 'cpu')
        else:
            W_rec = reconstruct_fn(q_entry, 'cpu')"""

content = content.replace(old_route, new_route)

# Fix the NameError
content = content.replace('reconstruct_from_svd_sub1bit', 'reconstruct_from_svd')

with open(eval_path, 'w') as f:
    f.write(content)

print("Patched script with unpack_ternary logic.")

# Run the evaluation
!python /content/quantization-exploration/scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --max-layers 10


Patched script with unpack_ternary logic.
RECONSTRUCTION QUALITY EVALUATION

[1] Loading original weights...
  Loaded 182 weight matrices
  Evaluating first 10 layers

[2] Evaluating reconstruction quality...

  Evaluating svd_60...
    Avg MSE: 0.921473
    Avg RMSE: 0.959934
    Max Error: 11.8074
    Estimated PPL: 465.74

  Evaluating svd_70...
    Avg MSE: 0.921499
    Avg RMSE: 0.959947
    Max Error: 11.8074
    Estimated PPL: 465.75

  Skipping quantized/gemma_magq.pt - not found

  Skipping quantized/gemma-4-E2B-sub1bit.pt - not found

SUMMARY - Ordered by Quality (best first)
Method                         Avg MSE      Max Error    Est. PPL    
----------------------------------------------------------------------
svd_60                         0.921473     11.8074      465.74      
svd_70                         0.921499     11.8074      465.75      
----------------------------------------------------------------------
Note: Estimated PPL assumes base Gemma 4 E2B perplexity